<a href="https://colab.research.google.com/github/satyaudaybandaru/Gemma-3-4B-FunctionCall/blob/main/Dataset_Preparation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Installation

In [ ]:
!pip install --upgrade unsloth transformers huggingface_hub datasets

<a name="Data"></a>
### Data Prep


In [ ]:
from datasets import load_dataset
datasets = load_dataset("Salesforce/APIGen-MT-5k",split='train')

In [ ]:
all_sys = datasets.unique('system')
print(len(all_sys))

#### Modify system messages to reduce the token count.

In [ ]:
new_sys = ['''# Airline Agent Policy

**Time:** 2024-05-15 15:00:00 EST

## Core Rules

* Handle only flight booking, modification, cancellation, and related support.
* Before any state-changing action (booking, flight/cabin/passenger/baggage updates, cancellation), summarize the changes and obtain explicit user confirmation ("yes").
* Use only user-provided information and tool results; do not provide external knowledge, recommendations, or opinions.
* Make at most one tool call per turn. Do not combine tool calls and user-facing responses.
* Reject requests that violate policy.
* Escalate to a human agent only when the request cannot be completed through supported actions.

## Data

* User profile: user ID, contact info, DOB, payment methods, reservations, membership tier.
* Reservation: reservation ID, user ID, trip details, passengers, payments, baggage, insurance.
* Flights:

  * **available** = bookable
  * **delayed/on time/flying** = not bookable

## Booking

* Collect: user ID, trip type, origin, destination.
* Max 5 passengers; collect first name, last name, DOB for each.
* All passengers must share the same flights and cabin.
* Payments: max 1 travel certificate, 1 credit card, 3 gift cards; all must exist in the user profile.
* Free checked bags per passenger:

  * Regular: Basic 0, Economy 1, Business 2
  * Silver: Basic 1, Economy 2, Business 3
  * Gold: Basic 2, Economy 3, Business 3
  * Extra bag: $50
* Offer travel insurance ($30/passenger).

## Modification

* Collect: user ID, reservation ID.
* Flight changes:

  * Basic Economy cannot be changed.
  * Origin, destination, and trip type cannot change.
  * Retained segments keep original pricing.
  * Verify eligibility before API calls.
* Cabin changes allowed for all reservations; user pays fare difference; cabin must match across all segments.
* Bags can be added, not removed.
* Insurance cannot be added after booking.
* Passenger details can change, but passenger count cannot.
* Flight changes require a credit card or gift card for payment/refund handling.

## Cancellation

* Collect: user ID, reservation ID, cancellation reason.
* Allowed if:

  * Within 24 hours of booking, or
  * Airline canceled the flight, or
  * Business class reservation, or
  * Basic/Economy reservation with valid travel insurance and eligible reason.
* Membership does not affect cancellation eligibility.
* Only entire unflown trips can be canceled.
* If any segment has been flown, escalate to a human agent.
* Refunds return to original payment methods in 5–7 business days.

## Compensation

Only if the user complains and explicitly requests compensation:

* Canceled flight: Silver/Gold member, insured traveler, or Business passenger → $100 × passengers.
* Delayed flight with cancellation/modification completed: same eligibility → $50 × passengers.
* No compensation for Regular members without insurance traveling in Basic Economy/Economy.
''',
'''# Retail Agent Policy

## Core Rules

* Authenticate the user at the start using:

  * email, or
  * name + ZIP code.
  * Never trust a provided user ID without authentication.
* After authentication, assist only that user for the remainder of the conversation.
* You may provide information about the authenticated user's profile, orders, and products.
* Before any state-changing action (cancel, modify, return, exchange), summarize the action and obtain explicit confirmation ("yes").
* Use only user-provided information and tool outputs; do not invent facts, procedures, or recommendations.
* Make at most one tool call per turn. Do not combine tool calls and user-facing responses.
* Escalate to a human agent only when the request is outside supported capabilities.

## Domain Rules

* All timestamps use EST (24-hour format).
* User profile: email, default address, user ID, payment methods (gift card, PayPal, credit card).
* Products have product IDs; variants/items have item IDs. Product IDs and item IDs are distinct.
* Order statuses: `pending`, `processed`, `delivered`, `cancelled`.
* Actions generally apply only to `pending` or `delivered` orders.
* Modify-item and exchange operations can only be executed once; collect all item changes before calling the tool.

## Cancel Pending Order

Requirements:

* Order status must be `pending`.
* Collect order ID and reason (`no longer needed` or `ordered by mistake`).
* After confirmation:

  * Status → `cancelled`
  * Refund to original payment method:

    * Gift card: immediate
    * Other methods: 5–7 business days

## Modify Pending Order

Requirements:

* Order status must be `pending`.
* Allowed modifications only:

  * shipping address
  * payment method
  * item options

### Payment Modification

* Must replace the original payment method with a single new payment method.
* Gift cards must have sufficient balance.
* After confirmation:

  * Order remains `pending`
  * Original payment refunded (immediate for gift cards, otherwise 5–7 business days)

### Item Modification

* Can only change an item to another variant of the same product.
* Product type cannot change.
* Collect all item modifications before execution and remind the user that no further modifications/cancellations will be possible afterward.
* User must provide a payment/refund method for price differences.
* Gift cards must have sufficient balance.
* After execution:

  * Status → `pending (items modified)`

## Return Delivered Order

Requirements:

* Order status must be `delivered`.
* Collect:

  * order ID
  * items to return
  * refund method
* Refund method must be:

  * original payment method, or
  * existing gift card.
* After confirmation:

  * Status → `return requested`
  * Return instructions sent via email.

## Exchange Delivered Order

Requirements:

* Order status must be `delivered`.
* Collect all items to exchange before execution.
* Exchanges are only between variants of the same product.
* Product type cannot change.
* User must provide a payment/refund method for price differences.
* Gift cards must have sufficient balance.
* After confirmation:

  * Status → `exchange requested`
  * Exchange instructions sent via email.
  * No new order is created.
''']

In [ ]:
print('Character Differences of Dataset System Prompts and Rewritten System Prompts\n')
for i in range(len(all_sys)):
  print(f'Dataset sys[{i}]: {len(all_sys[i])} - Rewritten: {len(new_sys[i])} - Percentage Reduced: {(1-(len(new_sys[i])/len(all_sys[i])))*100:.2f}%')

In [ ]:
old_sys_new = dict()
for old, new in zip(all_sys, new_sys):
  old_sys_new[old] = new

In [ ]:
import pprint
pprint.pprint(datasets[0])

#### Initialize New chat template

We now use the `Gemma-3` format for conversation style finetunes.

Gemma-3 original chat template:

```
<bos><start_of_turn>user
Hello!<end_of_turn>
<start_of_turn>model
Hey there!<end_of_turn>
```


In [ ]:
from unsloth import FastModel

model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-3-4b-it",
    max_seq_length = 4096, # Choose any for long context!
    load_in_4bit = True,  # 4 bit quantization to reduce memory
    load_in_8bit = False, # [NEW!] A bit more accurate, uses 2x memory
    full_finetuning = False, # [NEW!] We have full finetuning now!
    # token = "YOUR_HF_TOKEN", # HF Token for gated models
)

In [ ]:
tool_calls_template = '''{{ bos_token }}
{% macro render_content(content) %}
    {% if content is string %}
{{ content }}
    {% elif content is iterable %}
        {% for part in content %}
            {% if part["type"] == "text" %}
{{ part["text"] }}
            {% endif %}
        {% endfor %}
    {% endif %}
{% endmacro %}

{%- for message in messages %}
    {%- if message["role"] == "system" %}
<start_of_turn>system
{{ render_content(message["content"]) }}

        {%- if tools %}
You have access to the following tools:
<tools>
{{ tools | tojson }}
</tools>
        {%- endif %}
<end_of_turn>
    {%- endif %}
{%- endfor %}

{%- for message in messages %}

    {%- if message["role"] == "user" %}

<start_of_turn>user
{{ render_content(message["content"]) }}
<end_of_turn>

    {%- elif message["role"] == "assistant" and message.get("tool_calls") %}

<start_of_turn>model
<tool_call>
{{ message["tool_calls"] | map(attribute="function") | list | tojson }}
</tool_call>
<end_of_turn>

    {%- elif message["role"] == "tool" %}

<start_of_turn>tool
<tool_response>
{{ render_content(message["content"]) }}
</tool_response>
<end_of_turn>

    {%- elif message["role"] == "assistant" %}

<start_of_turn>model
{{ render_content(message["content"]) }}
<end_of_turn>

    {%- endif %}

{%- endfor %}

{% if add_generation_prompt %}
<start_of_turn>model
{% endif %}
'''

eos_token = tokenizer.eos_token

In [ ]:
tokenizer.eos_token

In [ ]:
from unsloth.chat_templates import get_chat_template
tk = get_chat_template(tokenizer, chat_template=(tool_calls_template, eos_token))
print(repr(tk.chat_template))

We now have to apply the chat template for `Gemma-3` onto the conversations, and save it to `text`. We remove the `<bos>` token using removeprefix(`'<bos>'`) since we're finetuning. The Processor will add this token before training and the model expects only one.

In [ ]:
import uuid
import json

def convert_to_unsloth_tool_format(examples):
    all_texts = []

    conversations = examples["conversations"]
    tools_list = examples.get("tools", [None] * len(conversations))
    systems = examples.get("system", [None] * len(conversations))

    for convo, tools_str, system_prompt in zip(conversations, tools_list, systems):

        templated_convo = []
        current_tool_call_id = None
        called_tools = []



        # ----------------------------
        # 1. SYSTEM (ONLY TEXT, NO TOOLS HERE)
        # ----------------------------
        if system_prompt:
            templated_convo.append({
                "role": "system",
                "content": [{"type":"text", "text": old_sys_new[system_prompt]}]
            })

        # ----------------------------
        # 2. PROCESS CONVERSATION
        # ----------------------------
        for turn in convo:

            raw_role = turn["from"]
            raw_value = turn["value"]

            # user
            if raw_role in ["human", "user"]:
                templated_convo.append({
                    "role": "user",
                    "content": [{"type":"text", "text": raw_value}]
                })

            # assistant normal
            elif raw_role in ["gpt", "assistant"]:
                templated_convo.append({
                    "role": "assistant",
                    "content": [{"type":"text", "text": raw_value}]
                })

            # tool call
            elif raw_role == "function_call":
                current_tool_call_id = f"call_{uuid.uuid4().hex[:8]}"

                try:
                    call_data = json.loads(raw_value)
                    called_tools.append(call_data.get("name"))
                    templated_convo.append({
                        "role": "assistant",
                        "content": None,
                        "tool_calls": [{
                            "id": current_tool_call_id,
                            "type": "function",
                            "function": {
                                "name": call_data.get("name"),
                                "arguments": call_data.get("arguments", {})
                            }
                        }]
                    })

                except Exception:
                    templated_convo.append({
                        "role": "assistant",
                        "content": [{"type":"text", "text": raw_value}]
                    })

            # tool response
            elif raw_role == "observation":

                if not current_tool_call_id:
                    current_tool_call_id = f"call_{uuid.uuid4().hex[:8]}"

                content = raw_value.strip() if isinstance(raw_value, str) else raw_value

                if content:
                    templated_convo.append({
                        "role": "tool",
                        "tool_call_id": current_tool_call_id,
                        "content": content
                    })

                current_tool_call_id = None

        # ----------------------------
        # 3. Parse tools
        # ----------------------------
        parsed_tools = None
        if tools_str:
            try:
                parsed_tools = json.loads(tools_str) if isinstance(tools_str, str) else tools_str
                filtered_tools = [tool for tool in parsed_tools if tool["name"] in called_tools]
                if len(filtered_tools) == 0:
                  filtered_tools = parsed_tools
                print(f'Removed {len(parsed_tools)-len(filtered_tools)} out of {len(parsed_tools)}')
            except Exception:
                filtered_tools = None


        # ----------------------------
        # 4. APPLY TEMPLATE
        # ----------------------------
        rendered_text = tk.apply_chat_template(
            templated_convo,
            tools=filtered_tools,   # IMPORTANT: ONLY HERE
            tokenize=False,
            add_generation_prompt=False
        ).removeprefix('<bos>')

        all_texts.append(rendered_text)

    return {"text": all_texts}


dataset = datasets.map(convert_to_unsloth_tool_format, batched=True)

In [ ]:
tokenized = []
token_lengths = []
for row in dataset.select(range(100)):
  temp = tk(row['text'], add_generation_prompt=False, tokenize=True)
  tokenized.append(temp)
  token_lengths.append(len(temp.input_ids[0]))

In [ ]:
f'Highest Tokens: {max(token_lengths)} at {token_lengths.index(max(token_lengths))}'

Let's see how the chat template did! Notice there is no `<bos>` token as the processor tokenizer will be adding one.

In [ ]:
# Check random datapoint
dataset[98]['text']

In [ ]:
tk.decode(tokenized[98].input_ids[0][:2048])

In [ ]:
print(len(tk(dataset[11]['text']).input_ids[0]))

In [ ]:
def print_ind_tokens(row):
  tools_start = dataset[row]['text'].index('You have access to the following tools:')
  tools_end = dataset[row]['text'].index('<start_of_turn>user')
  system = dataset[row]['text'][:tools_start]
  tools = dataset[row]['text'][tools_start:tools_end]
  convo = dataset[row]['text'][tools_end:]

  sys_tok = len(tk(system).input_ids[0])
  tools_tok = len(tk(tools).input_ids[0])
  convo_tok = len(tk(convo).input_ids[0])
  print(f'For Row-{row}\nSystem: {sys_tok} Tokens\nTools: {tools_tok} Tokens\nConversations: {convo_tok} Tokens\nSystem+tools: {sys_tok+tools_tok}\n\n')

Verify the token count for each data point.

Keep conversations within the 4096-token context limit and avoid excessive allocation to system and tool tokens.

In [ ]:
for i in range(100):
  print_ind_tokens(i)

In [ ]:
import numpy as np
check_all_tokenized = tk(list(dataset['text']))

In [ ]:
import numpy as np
all_token_lengths = [len(ids) for ids in check_all_tokenized.input_ids]
print(f'Max: {max(all_token_lengths)}')
print(f'50th percentile: {np.percentile(all_token_lengths, 50)}')
print(f'60th percentile: {np.percentile(all_token_lengths, 60)}')
print(f'75th percentile: {np.percentile(all_token_lengths, 75)}')
print(f'90th percentile: {np.percentile(all_token_lengths, 90)}')

We can't process more than 4096 sequences in free tier, so filter the rows > 4000 tokens

In [ ]:
np.sum(np.array(all_token_lengths) < 4000)

In [ ]:
filtered_indices = np.where(np.array(all_token_lengths)< 4000)
filtered_dataset = dataset.select(filtered_indices[0])

In [ ]:
#check random datapoint
filtered_dataset[99]['text']

## Save Dataset to Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
filtered_dataset.save_to_disk("/content/drive/MyDrive/gemma3-dataset")

**Continue to Phase-1 Finetuning**